# agentic-civic-resolution-app 
Agent 2: Severity Scoring Agent

**Role:** Complaint text + classification → severity score (1–5) + priority label + SLA hours

Severity drives downstream SLA, escalation paths, and dashboard alerting.

 | Score | Label    | SLA       | Examples                                    |
 |-------|----------|-----------|---------------------------------------------|
 | 5     | Critical | 4 hrs     | Exposed live wires, sewage flood, fire risk |
 | 4     | High     | 24 hrs    | Major pothole, water main burst             |
 | 3     | Medium   | 72 hrs    | Garbage overflow, streetlight out           |
 | 2     | Low      | 7 days    | Littering, minor noise, overgrown hedge     |
 | 1     | Minimal  | 30 days   | Cosmetic damage, information requests       |

**COMMAND**

 %pip install databricks-langchain mlflow pydantic --quiet

 dbutils.library.restartPython()

In [0]:
%pip install databricks-langchain mlflow pydantic --quiet

#dbutils.library.restartPython()

## 0. Config & Imports

In [0]:
import json, re, mlflow, requests
from pydantic import BaseModel, Field
from databricks_langchain import ChatDatabricks

def ensure_mlflow_experiment(experiment_path: str) -> str:
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    parts = experiment_path.strip("/").split("/")
    for depth in range(1, len(parts)):
        folder = "/" + "/".join(parts[:depth])
        resp   = requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=headers, json={"path": folder})
        body   = resp.json()
        if resp.status_code == 200:
            print(f"  ✓ folder: {folder}")
        elif body.get("error_code") == "RESOURCE_ALREADY_EXISTS":
            print(f"  ✓ exists: {folder}")
        else:
            raise RuntimeError(f"mkdirs failed for {folder}: {body}")

    exp = mlflow.set_experiment(experiment_path)
    return exp.experiment_id

EXPERIMENT_ID = ensure_mlflow_experiment("/civicops/agents/severity_scorer")
print(f"✓ MLflow experiment ready (id={EXPERIMENT_ID})")

LLM_ENDPOINT = "databricks-meta-llama-3-1-70b-instruct"

# SLA lookup by score
SLA_HOURS = {5: 4, 4: 24, 3: 72, 2: 168, 1: 720}
PRIORITY_LABELS = {5: "Critical", 4: "High", 3: "Medium", 2: "Low", 1: "Minimal"}

In [0]:
def discover_llm_endpoint() -> str:
    ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host    = ctx.apiUrl().get()
    token   = ctx.apiToken().get()
    headers = {"Authorization": f"Bearer {token}"}
    resp    = requests.get(f"{host}/api/2.0/serving-endpoints", headers=headers)
    ready   = [e for e in resp.json().get("endpoints", []) if e.get("state", {}).get("ready") == "READY"]
    if not ready:
        raise RuntimeError("No READY serving endpoints found. Go to Serving → create one first.")
    print("Available READY endpoints:")
    for e in ready:
        print(f"  • {e['name']}")
    for kw in ["llama", "dbrx", "mixtral", "mpt"]:
        for e in ready:
            if kw in e["name"].lower():
                print(f"✓ Auto-selected: {e['name']}")
                return e["name"]
    return ready[0]["name"]

LLM_ENDPOINT = discover_llm_endpoint()

## 1. Output Schema

In [0]:
class SeverityOutput(BaseModel):
    severity_score:   int   = Field(ge=1, le=5,  description="Severity 1 (minimal) to 5 (critical)")
    priority_label:   str   = Field(description="Critical / High / Medium / Low / Minimal")
    sla_hours:        int   = Field(description="Expected resolution time in hours")
    health_risk:      bool  = Field(description="True if complaint poses public health or safety risk")
    infrastructure_risk: bool = Field(description="True if critical infrastructure is at risk")
    affected_estimate: str  = Field(description="Rough estimate: 'individual', '10-50 people', '100+ people'")
    reasoning:        str   = Field(description="Two sentences justifying the score")

## 2. Scoring Rules & System Prompt

In [0]:
SYSTEM_PROMPT = """
You are the Severity Scoring Agent for CivicOps AI.

Given a civic complaint (text + optional classification), score its severity from 1 to 5.

## Severity Rubric

| Score | Label    | SLA    | Criteria                                                                   |
|-------|----------|--------|----------------------------------------------------------------------------|
| 5     | Critical | 4 hrs  | Immediate danger to life/infrastructure: exposed wires, sewage flood, fire |
| 4     | High     | 24 hrs | Significant disruption: water main burst, major pothole, road blocked       |
| 3     | Medium   | 72 hrs | Moderate impact: garbage overflow, streetlight out, stray animals           |
| 2     | Low      | 7 days | Minor inconvenience: littering, noise, overgrown hedge                      |
| 1     | Minimal  | 30 days| Cosmetic or informational: faded paint, info request                        |

## Boosting Signals (increase score by 1 if present)
- Keywords: "children", "hospital", "school", "elderly", "flood", "fire", "accident"
- Scale words: "entire street", "whole building", "neighbourhood"
- Urgency words: "since days", "weeks", "no response"

## Output JSON Schema (respond ONLY with this, no markdown):
{
  "severity_score": <int 1-5>,
  "priority_label": "<Critical|High|Medium|Low|Minimal>",
  "sla_hours": <int>,
  "health_risk": <true|false>,
  "infrastructure_risk": <true|false>,
  "affected_estimate": "<individual|10-50 people|100+ people>",
  "reasoning": "<two sentences>"
}
""".strip()

## 3. Agent Class

In [0]:
class SeverityScorerAgent:
    """Scores severity of a civic complaint using LLM + rule overrides."""

    def __init__(self, endpoint: str = LLM_ENDPOINT):
        self.llm      = ChatDatabricks(endpoint=endpoint, temperature=0.0)
        self.endpoint = endpoint

    def score(
        self,
        complaint_text: str,
        category: str | None = None,
        subcategory: str | None = None,
    ) -> SeverityOutput:
        """
        Args:
            complaint_text: Raw citizen complaint.
            category:       Output from ClassifierAgent (optional but improves accuracy).
            subcategory:    Output from ClassifierAgent (optional).
        """
        context_block = ""
        if category:
            context_block = f"\nClassification: {category} > {subcategory or 'N/A'}"

        user_msg = f"Complaint: {complaint_text.strip()}{context_block}"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ]

        with mlflow.start_run(run_name="score_severity", nested=True):
            mlflow.log_params({"endpoint": self.endpoint, "input": complaint_text[:200]})
            response = self.llm.invoke(messages)
            result   = self._parse(response.content)

            # Rule-based override: force SLA to match score table
            result.sla_hours      = SLA_HOURS[result.severity_score]
            result.priority_label = PRIORITY_LABELS[result.severity_score]

            mlflow.log_metric("severity_score", result.severity_score)
            mlflow.log_metric("health_risk",    int(result.health_risk))

        return result

    def score_batch(
        self,
        complaints: list[str],
        categories: list[str | None] | None = None,
        subcategories: list[str | None] | None = None,
    ) -> list[SeverityOutput]:
        cats  = categories    or [None] * len(complaints)
        subs  = subcategories or [None] * len(complaints)
        return [self.score(c, ca, su) for c, ca, su in zip(complaints, cats, subs)]

    @staticmethod
    def _parse(raw: str) -> SeverityOutput:
        raw = raw.strip()
        try:
            return SeverityOutput(**json.loads(raw))
        except Exception:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                return SeverityOutput(**json.loads(m.group()))
            raise ValueError(f"Unparseable severity output:\n{raw}")


severity_agent = SeverityScorerAgent()
print("✓ Severity scoring agent ready.")

## 4. Smoke Test

In [0]:
_tests = [
    ("Garbage overflow near Whitefield",                       "Sanitation",    "Garbage Overflow"),
    ("Exposed live electrical wires near the playground",      "Electricity",   "Exposed Wires"),
    ("Small crack in the footpath outside my house",           "Infrastructure","Sidewalk Damage"),
    ("Water pipe burst — entire street is flooded since 2 days","Water & Sewage","Water Leakage"),
    ("Faded road markings near the junction",                  "Road & Traffic","Missing Road Markings"),
]

print(f"{'Complaint':<52} Score  Priority   SLA(h)  Health  Infra")
print("-" * 100)
for text, cat, sub in _tests:
    r = severity_agent.score(text, cat, sub)
    print(
        f"{text[:51]:<52} "
        f"{r.severity_score}      "
        f"{r.priority_label:<10} "
        f"{r.sla_hours:<7} "
        f"{'Y' if r.health_risk else 'N':<7} "
        f"{'Y' if r.infrastructure_risk else 'N'}"
    )

## 5. Register in MLflow Model Registry

In [0]:
import pandas as pd
from mlflow.models.signature import infer_signature

class _SeverityWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.agent = SeverityScorerAgent()

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in model_input.iterrows():
            r = self.agent.score(
                str(row["complaint_text"]),
                category=row.get("category"),
                subcategory=row.get("subcategory"),
            )
            results.append(r.model_dump())
        return pd.DataFrame(results)

_input_example  = pd.DataFrame({"complaint_text": ["Garbage overflow"], "category": ["Sanitation"], "subcategory": ["Garbage Overflow"]})
_output_example = pd.DataFrame([SeverityOutput(
    severity_score=3, priority_label="Medium", sla_hours=72,
    health_risk=False, infrastructure_risk=False,
    affected_estimate="10-50 people", reasoning="Example."
).model_dump()])
_signature = infer_signature(_input_example, _output_example)

spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.agents")
with mlflow.start_run(run_name="register_severity"):
    mlflow.pyfunc.log_model(
        name="severity_scorer",
        python_model=_SeverityWrapper(),
        pip_requirements=["databricks-langchain", "pydantic"],
        input_example=_input_example,
        signature=_signature,
        registered_model_name="civicops.agents.severity_scorer",
    )
print("✓ Severity agent registered in MLflow Model Registry.")